In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv("/content/AdventureWorks_Dataset.csv")
conn = sqlite3.connect(":memory:")
df.to_sql("sales", conn, index=False, if_exists="replace")
def q(sql):
    return pd.read_sql_query(sql, conn)

print(df.shape)
df.head()

(11124, 28)


,SalesOrderDetailID,SalesOrderID,OrderDate,CustomerID,CustomerName,City,TerritoryID,TerritoryName,CountryRegion,GroupName,...,StandardCost,ListPrice,OrderQty,UnitPrice,UnitPriceDiscount,LineTotal,SubTotal,TaxAmt,Freight,TotalDue
0,1,1,2024-01-01,380,Rachel Tsoflias,Phoenix,9,United Kingdom,United Kingdom,Europe,...,2081.71,2966.41,1,2966.41,0.1,2669.77,4061.36,335.06,101.53,4497.95
1,2,1,2024-01-01,380,Rachel Tsoflias,Phoenix,9,United Kingdom,United Kingdom,Europe,...,860.07,1391.59,1,1391.59,0.0,1391.59,4061.36,335.06,101.53,4497.95
2,3,2,2024-01-01,268,Tsvi Carson,Boston,6,Canada,Canada,North America,...,1191.43,1655.91,3,1655.91,0.0,4967.73,5335.98,440.22,133.40,5909.60
3,4,2,2024-01-01,268,Tsvi Carson,Boston,6,Canada,Canada,North America,...,76.70,126.65,1,126.65,0.0,126.65,5335.98,440.22,133.40,5909.60
4,5,2,2024-01-01,268,Tsvi Carson,Boston,6,Canada,Canada,North America,...,42.03,60.40,4,60.40,0.0,241.60,5335.98,440.22,133.40,5909.60


In [3]:
q("""
SELECT
    CustomerName,
    OrderDate,
    LineTotal,
    ROW_NUMBER() OVER (PARTITION BY CustomerName ORDER BY OrderDate) AS rn
FROM sales
LIMIT 10
""")  #Query for testing before starting

,CustomerName,OrderDate,LineTotal,rn
0,Amy Ansman-Wolfe,2024-02-18,151.06,1
1,Amy Ansman-Wolfe,2024-03-20,319.09,2
2,Amy Ansman-Wolfe,2024-03-20,91.20,3
3,Amy Ansman-Wolfe,2024-03-20,4997.14,4
4,Amy Ansman-Wolfe,2024-12-31,230.93,5
5,Amy Ansman-Wolfe,2024-12-31,9994.28,6
6,Amy Ansman-Wolfe,2024-12-31,2783.18,7
7,Amy Ansman-Wolfe,2024-12-31,9.83,8
8,Amy Ansman-Wolfe,2025-02-22,2783.18,9
9,Amy Ansman-Wolfe,2025-02-22,253.30,10


In [4]:
# Q1: ROW_NUMBER() - Numbers each customer's orders 1, 2, 3... in date order
q("""
SELECT CustomerName, SalesOrderID, OrderDate, TotalDue,
       ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY OrderDate, SalesOrderID) AS order_no
FROM (SELECT DISTINCT SalesOrderID, OrderDate, CustomerID, CustomerName, TotalDue FROM sales)
ORDER BY CustomerID, order_no
LIMIT 20
""")

,CustomerName,SalesOrderID,OrderDate,TotalDue,order_no
0,Syed Jiang,11,2024-01-03,321.20,1
1,Syed Jiang,186,2024-02-12,146.33,2
2,Syed Jiang,203,2024-02-15,842.13,3
3,Syed Jiang,574,2024-05-10,5687.58,4
4,Syed Jiang,773,2024-06-25,16517.11,5
5,Syed Jiang,1037,2024-08-22,8135.48,6
6,Syed Jiang,1534,2024-12-15,1948.44,7
7,Syed Jiang,1556,2024-12-20,3020.73,8
8,Syed Jiang,2121,2025-04-27,601.57,9
9,Syed Jiang,2204,2025-05-16,14002.13,10


In [5]:
# Q2: ROW_NUMBER() - Finds the top 3 best-selling products inside each category by revenue
q("""
SELECT * FROM (
  SELECT Category, ProductName, ROUND(SUM(LineTotal),2) AS revenue,
         ROW_NUMBER() OVER (PARTITION BY Category ORDER BY SUM(LineTotal) DESC) AS rn
  FROM sales GROUP BY Category, ProductName
) WHERE rn <= 3
ORDER BY Category, rn
""")

,Category,ProductName,revenue,rn
0,Accessories,Pro Helmet,65941.34,1
1,Accessories,Sport-200 Helmet,45005.47,2
2,Accessories,Sport-100 Helmet,38842.56,3
3,Bikes,Road-750,2353371.79,1
4,Bikes,Mountain-300,2302925.31,2
5,Bikes,Mountain-100,2152443.09,3
6,Clothing,Racing Shorts,81555.24,1
7,Clothing,Bib-Shorts,68844.14,2
8,Clothing,Classic Jersey,64885.96,3
9,Components,HL Wheel,208703.03,1


In [6]:
# Q3: RANK() - Ranks all products by total revenue equal revenue gets the same rank, next rank is skipped.
q("""
SELECT ProductName, Category, ROUND(SUM(LineTotal),2) AS revenue,
       RANK() OVER (ORDER BY SUM(LineTotal) DESC) AS revenue_rank
FROM sales GROUP BY ProductName, Category
ORDER BY revenue_rank
LIMIT 15
""")

,ProductName,Category,revenue,revenue_rank
0,Road-750,Bikes,2353371.79,1
1,Mountain-300,Bikes,2302925.31,2
2,Mountain-100,Bikes,2152443.09,3
3,Mountain-500,Bikes,1665157.28,4
4,Road-550,Bikes,1581650.72,5
5,Touring-3000,Bikes,1390368.36,6
6,Mountain-200,Bikes,1346051.06,7
7,Road-150,Bikes,1279417.84,8
8,Touring-1000,Bikes,1264621.03,9
9,Touring-2000,Bikes,1104618.83,10


In [7]:
# Q4: ROW_NUMBER vs RANK vs DENSE_RANK - Compares the 3 ranking functions on customers with tied order counts
# ROW_NUMBER = always unique , RANK = ties share rank, gaps after - DENSE_RANK = ties share rank, no gaps
q("""
SELECT CustomerName, COUNT(DISTINCT SalesOrderID) AS total_orders,
       ROW_NUMBER() OVER (ORDER BY COUNT(DISTINCT SalesOrderID) DESC) AS row_num,
       RANK()       OVER (ORDER BY COUNT(DISTINCT SalesOrderID) DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY COUNT(DISTINCT SalesOrderID) DESC) AS dense_rnk
FROM sales GROUP BY CustomerID, CustomerName
ORDER BY row_num
LIMIT 15
""")

,CustomerName,total_orders,row_num,rnk,dense_rnk
0,Amy Mitchell,20,1,1,1
1,Shu Mensa-Annan,20,2,1,1
2,Tsvi Valdez,18,3,3,2
3,Carol Campbell,18,4,3,2
4,Syed Blythe,18,5,3,2
5,Linda Campbell,17,6,6,3
6,Jose Varkey Chudukatil,17,7,6,3
7,Syed Jiang,16,8,8,4
8,Stephen Tsoflias,16,9,8,4
9,Tsvi Jiang,16,10,8,4


In [8]:
# Q5: RANK() with PARTITION BY - Finds the top 3 customers by revenue in each sales territory
q("""
SELECT * FROM (
  SELECT TerritoryName, CustomerName, ROUND(SUM(LineTotal),2) AS revenue,
         RANK() OVER (PARTITION BY TerritoryName ORDER BY SUM(LineTotal) DESC) AS rnk
  FROM sales GROUP BY TerritoryName, CustomerID, CustomerName
) WHERE rnk <= 3
ORDER BY TerritoryName, rnk
""")

,TerritoryName,CustomerName,revenue,rnk
0,Australia,Shu Campbell,106418.27,1
1,Australia,Lynn Ito,89326.65,2
2,Australia,Amy Mitchell,81546.08,3
3,Canada,Syed Saraiva,92937.27,1
4,Canada,Syed Blythe,86748.72,2
5,Canada,Linda Varkey Chudukatil,85833.47,3
6,Central,Stephen Tsoflias,91455.57,1
7,Central,Lynn Carson,89093.87,2
8,Central,Carol Abbas,75913.37,3
9,France,Jillian Mitchell,89564.72,1


In [9]:
# Q6: RANK() - Ranks salespeople by total sales and shows how much of their quota they achieved
q("""
SELECT SalesPersonName, SalesQuota, ROUND(SUM(LineTotal),2) AS total_sales,
       ROUND(SUM(LineTotal)*100.0/SalesQuota,1) AS pct_of_quota,
       RANK() OVER (ORDER BY SUM(LineTotal) DESC) AS sales_rank
FROM sales GROUP BY SalesPersonID, SalesPersonName, SalesQuota
ORDER BY sales_rank
""")

,SalesPersonName,SalesQuota,total_sales,pct_of_quota,sales_rank
0,Jose Carson,750000.0,2116295.67,282.2,1
1,Tete Ansman-Wolfe,750000.0,2028981.35,270.5,2
2,Garrett Alberts,500000.0,2026061.42,405.2,3
3,Carol Ito,500000.0,1924046.19,384.8,4
4,Stephen Ansman-Wolfe,650000.0,1886603.74,290.2,5
5,Jose Ansman-Wolfe,500000.0,1862721.91,372.5,6
6,Tete Ito,300000.0,1859337.37,619.8,7
7,Carol Jiang,250000.0,1844451.68,737.8,8
8,Michael Vargas,500000.0,1721952.22,344.4,9
9,Linda Campbell,750000.0,1697545.81,226.3,10


In [10]:
# Q7: LAG() - Brings the previous month's revenue into the current row to calculate Month-over-Month growth %
# First row shows NaN because there is no earlier month
q("""
WITH monthly AS (
  SELECT SUBSTR(OrderDate,1,7) AS month, ROUND(SUM(LineTotal),2) AS revenue
  FROM sales GROUP BY 1
)
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
       ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0
             / LAG(revenue) OVER (ORDER BY month), 2) AS mom_growth_pct
FROM monthly
ORDER BY month
""")

,month,revenue,prev_month_revenue,mom_growth_pct
0,2024-01,629742.11,NaN,NaN
1,2024-02,442282.02,629742.11,-29.77
2,2024-03,729931.29,442282.02,65.04
3,2024-04,565972.42,729931.29,-22.46
4,2024-05,597810.96,565972.42,5.63
5,2024-06,503120.69,597810.96,-15.84
6,2024-07,606630.82,503120.69,20.57
7,2024-08,568730.52,606630.82,-6.25
8,2024-09,493882.80,568730.52,-13.16
9,2024-10,597656.84,493882.80,21.01


In [11]:
# Q8: LAG() with PARTITION BY - Gets each customer's previous order date to calculate days between two orders
q("""
SELECT CustomerName, OrderDate,
       LAG(OrderDate) OVER (PARTITION BY CustomerID ORDER BY OrderDate, SalesOrderID) AS prev_order_date,
       CAST(julianday(OrderDate) - julianday(
            LAG(OrderDate) OVER (PARTITION BY CustomerID ORDER BY OrderDate, SalesOrderID)) AS INT) AS days_gap
FROM (SELECT DISTINCT SalesOrderID, OrderDate, CustomerID, CustomerName FROM sales)
ORDER BY CustomerID, OrderDate
LIMIT 20
""")

,CustomerName,OrderDate,prev_order_date,days_gap
0,Syed Jiang,2024-01-03,None,NaN
1,Syed Jiang,2024-02-12,2024-01-03,40.0
2,Syed Jiang,2024-02-15,2024-02-12,3.0
3,Syed Jiang,2024-05-10,2024-02-15,85.0
4,Syed Jiang,2024-06-25,2024-05-10,46.0
5,Syed Jiang,2024-08-22,2024-06-25,58.0
6,Syed Jiang,2024-12-15,2024-08-22,115.0
7,Syed Jiang,2024-12-20,2024-12-15,5.0
8,Syed Jiang,2025-04-27,2024-12-20,128.0
9,Syed Jiang,2025-05-16,2025-04-27,19.0


In [12]:
# Q9: LAG() with PARTITION BY - Compares each category's yearly revenue with the previous year
q("""
WITH yearly AS (
  SELECT Category, SUBSTR(OrderDate,1,4) AS year, ROUND(SUM(LineTotal),2) AS revenue
  FROM sales GROUP BY Category, year
)
SELECT Category, year, revenue,
       LAG(revenue) OVER (PARTITION BY Category ORDER BY year) AS prev_year_revenue,
       ROUND((revenue - LAG(revenue) OVER (PARTITION BY Category ORDER BY year)) * 100.0
             / LAG(revenue) OVER (PARTITION BY Category ORDER BY year), 2) AS yoy_growth_pct
FROM yearly
ORDER BY Category, year
""")

,Category,year,revenue,prev_year_revenue,yoy_growth_pct
0,Accessories,2024,80300.78,NaN,NaN
1,Accessories,2025,85788.86,80300.78,6.83
2,Accessories,2026,60218.83,85788.86,-29.81
3,Bikes,2024,6245384.07,NaN,NaN
4,Bikes,2025,6383083.67,6245384.07,2.20
5,Bikes,2026,4750509.89,6383083.67,-25.58
6,Clothing,2024,154670.33,NaN,NaN
7,Clothing,2025,161216.20,154670.33,4.23
8,Clothing,2026,114279.32,161216.20,-29.11
9,Components,2024,333339.73,NaN,NaN


In [13]:
# Q10: SUM() OVER() - Running total: adds up revenue month by month to show cumulative growth
q("""
WITH monthly AS (
  SELECT SUBSTR(OrderDate,1,7) AS month, SUM(LineTotal) AS revenue FROM sales GROUP BY 1
)
SELECT month, ROUND(revenue,2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY month),2) AS running_total
FROM monthly
ORDER BY month
""")

,month,revenue,running_total
0,2024-01,629742.11,629742.11
1,2024-02,442282.02,1072024.13
2,2024-03,729931.29,1801955.42
3,2024-04,565972.42,2367927.84
4,2024-05,597810.96,2965738.80
5,2024-06,503120.69,3468859.49
6,2024-07,606630.82,4075490.31
7,2024-08,568730.52,4644220.83
8,2024-09,493882.80,5138103.63
9,2024-10,597656.84,5735760.47


In [14]:
# Q11: AVG() OVER() with window frame - 3-month moving average that smooths out monthly ups and downs
# ROWS BETWEEN 2 PRECEDING AND CURRENT ROW = current month + previous 2 months
q("""
WITH monthly AS (
  SELECT SUBSTR(OrderDate,1,7) AS month, SUM(LineTotal) AS revenue FROM sales GROUP BY 1
)
SELECT month, ROUND(revenue,2) AS revenue,
       ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),2) AS moving_avg_3m
FROM monthly
ORDER BY month
""")

,month,revenue,moving_avg_3m
0,2024-01,629742.11,629742.11
1,2024-02,442282.02,536012.06
2,2024-03,729931.29,600651.81
3,2024-04,565972.42,579395.24
4,2024-05,597810.96,631238.22
5,2024-06,503120.69,555634.69
6,2024-07,606630.82,569187.49
7,2024-08,568730.52,559494.01
8,2024-09,493882.80,556414.71
9,2024-10,597656.84,553423.39


In [15]:
# Q12: ROW_NUMBER() = 1 - Picks only the first order of every customer (common trick to get "first row per group")
q("""
SELECT * FROM (
  SELECT CustomerName, SalesOrderID, OrderDate, TotalDue,
         ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY OrderDate, SalesOrderID) AS rn
  FROM (SELECT DISTINCT SalesOrderID, OrderDate, CustomerID, CustomerName, TotalDue FROM sales)
) WHERE rn = 1
ORDER BY OrderDate
LIMIT 20
""")

,CustomerName,SalesOrderID,OrderDate,TotalDue,rn
0,Tsvi Carson,2,2024-01-01,5909.60,1
1,Lynn Saraiva,3,2024-01-01,298.43,1
2,Rachel Tsoflias,1,2024-01-01,4497.95,1
3,Amy Saraiva,6,2024-01-02,302.00,1
4,Ranjit Ansman-Wolfe,5,2024-01-02,2870.36,1
5,Jose Campbell,4,2024-01-02,9533.92,1
6,Syed Jiang,11,2024-01-03,321.20,1
7,Tsvi Mensa-Annan,10,2024-01-03,11668.10,1
8,Carol Abbas,7,2024-01-03,26597.37,1
9,Pamela Blythe,12,2024-01-03,7436.48,1


# Conclusion: SQL Window Functions Intro

## Objective
The goal of this task was to learn analytical SQL patterns using window functions (ROW_NUMBER, RANK, LAG) on an AdventureWorks-style sales dataset, and to use them to answer real business questions.

## Dataset
- 11,124 order lines across 4,480 orders (Jan 2024 to Sep 2026)
- 500 customers, 38 products, 10 territories, 10 salespeople
- Loaded into SQLite through Google Colab as a single table called `sales`

## What I Learned

| Function | Purpose | Business Use |
|---|---|---|
| ROW_NUMBER() | Gives every row a unique number | Top N per group, first order per customer |
| RANK() | Same rank for ties, next rank is skipped | Ranking products, customers, salespeople |
| DENSE_RANK() | Same rank for ties, no rank is skipped | Ranking without gaps |
| LAG() | Reads the previous row's value | Month-over-month and year-over-year growth, gap between orders |
| SUM() OVER() | Running total | Cumulative revenue |
| AVG() OVER() | Moving average | Smoothing monthly trends |
| PARTITION BY | Restarts the calculation for each group | Per customer, per category, per territory |

## Key Findings
- **Top product:** Road-750 was the highest revenue product (about 2.35M), closely followed by Mountain-300 (about 2.30M).
- **Top salesperson:** Jose Carson ranked first with about 2.12M in sales, followed by Tete Ansman-Wolfe with about 2.03M.
- **Ties in rankings:** Several customers had the same number of orders (20). ROW_NUMBER gave them different numbers, while RANK and DENSE_RANK gave them the same rank. This showed the difference between the three functions.
- **Revenue trend:** Monthly revenue moved up and down (for example, February 2024 dropped about 29.8% from January 2024). The 3-month moving average helped smooth these swings.
- **Category growth:** LAG made it easy to compare each category with the previous year. For example, Accessories grew about 6.8% from 2024 to 2025.

## Challenges and Notes
- The flat table repeats order-level values (SubTotal, TotalDue) on every line of the same order. I used **SELECT DISTINCT** for order-level analysis and **LineTotal **for revenue to avoid double counting.
- LAG returns NULL/NaN for the first row of each group because no earlier row exists.
- The dataset is synthetic and simplified, so the quota percentages are illustrative and not real business results.

## Final Takeaway
Window functions let me calculate rankings, trends and running totals **without collapsing rows** like GROUP BY does. They are simpler and cleaner than self-joins or subqueries, and they are very useful for real analytical work such as finding top performers, tracking growth and studying customer behavior.